# Filled posts models: out-of-fold evaluation (experiment, ticket 2166)

Scores the production registry models on held-out locations, against **known** ASC-WDS values only.
Ticket 2167. Built to be re-run as the template for experiments (plans 3 and 4).

**Run order:** settings, then top to bottom. Set `DATA_BUCKET` and `RESULTS_PREFIX` first.

**Caveats (read before interpreting results)**
- The rolling-average features are built from every location, including held-out ones. Only matters in small groups (checked in section 2).
- `non_res_without_dormancy_model` is frozen in production. Here it is retrained on four folds, so the baseline is a retrain, trained on data before 2025.
- Known = update events: the first month of each run of identical values (`01_filled_posts_02_cleaned`). Predicted and known are scored on the same rows, so known totals are not stock totals.
- Non-res size bands come from each location's mean known posts, so they are outcome-based: small bands tend to look over-predicted and large bands under-predicted for any model.
- The combined non-res model has no prediction where the dormancy model has none and the without-dormancy model has none. Those rows are dropped and the coverage reported.
- Training is as production: known target and `care_home_status_count == 1`. The non-res target includes PIR-based values for locations with no ASC-WDS.

## 0. Setup (SageMaker only)

Run the next two cells once, then restart the kernel. If `pip` fails because Python is older than 3.11 or there is no internet, stop and paste the error.

In [ ]:
%pip install polars==1.44.1 scikit-learn==1.7.1 matplotlib pyarrow

In [ ]:
!rm -rf repo repo-2166
!git clone --depth 1 --branch 2166-fp-exp-1 https://github.com/NMDSdevopsServiceAdm/DataEngineering.git repo-2166
!git -C repo-2166 rev-parse HEAD

## 1. Settings

In [ ]:
# Mode 1, experiment:         VARIANT = "<name>", RUN_LABEL = VARIANT, BASELINE_RUN_LABEL = "baseline"
# Mode 2, second-seed rescore: RUN_LABEL = f"{VARIANT}_seed43", BASELINE_RUN_LABEL = "baseline_seed43"
# Mode 3, second-seed baseline: VARIANT = "none", RUN_LABEL = "baseline_seed43", BASELINE_RUN_LABEL = None, FOLD_SEED = 43
VARIANT = "none"  # a key of VARIANTS in the experiment cell below
RUN_LABEL = VARIANT
BASELINE_RUN_LABEL = "baseline"  # folds and jitter sample are reused from this run

DATA_BUCKET = "sfc-2167-model-eval-datasets"  # read-only; confirm the name. Never push to this data branch
RESULTS_PREFIX = "s3://skillsforcare/emr-notebooks/filled-posts-eval/"  # persistent, not a branch bucket

CACHE_LOCATIONS = True
RUN_BRIDGE = False
FIRST_FY = None
FOLD_SEED = 42  # only used when BASELINE_RUN_LABEL is None
N_FOLDS = 5
JITTER_SAMPLE_SIZE = 10_000
SMALL_GROUP_LOCATIONS = 30

LOCAL_ROOT = None


In [ ]:
import hashlib
import io
import json
import platform
import subprocess
import sys
import time
from datetime import date, datetime, timezone
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import sklearn

REPO_DIR = Path("repo-2166").resolve()
sys.path.insert(0, str(REPO_DIR))

from polars_utils import utils as polars_utils
from polars_utils.expressions import is_care_home, is_not_care_home
from projects._03_independent_cqc._01_filled_posts._04_model.registry.model_registry import (
    model_registry,
)
from projects._03_independent_cqc._01_filled_posts._04_model.utils import (
    cross_validation_utils as cv,
)
from projects._03_independent_cqc._01_filled_posts._04_model.utils import (
    model_utils,
    paths,
    versioning,
)
from projects._03_independent_cqc._01_filled_posts._05_workplace_estimates.fargate.utils.models import (
    non_res_with_and_without_dormancy_combined as combine,
)
from projects._03_independent_cqc._01_filled_posts._05_workplace_estimates.fargate.utils.models import (
    utils as model_join,
)
from projects._03_independent_cqc.utils import model_evaluation_utils as evaluation
from utils.column_names.ind_cqc_pipeline_columns import IndCqcColumns as IndCQC
from utils.column_names.ind_cqc_pipeline_columns import ModelEvaluationColumns as Eval
from utils.column_names.ind_cqc_pipeline_columns import ModelRegistryKeys as MRKeys
from utils.column_values.categorical_column_values import AscwdsFilteringRule

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(220)
pl.Config.set_fmt_str_lengths(60)

COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
try:
    import psutil

    memory = f"{psutil.virtual_memory().total / 2**30:.0f} GiB"
except ImportError:
    memory = "unknown (psutil not installed)"
print(f"python {platform.python_version()} on {platform.platform()}")
print(f"polars {pl.__version__}, scikit-learn {sklearn.__version__}, numpy {np.__version__}")
print(f"memory {memory}; repo commit {COMMIT or 'not found'}")
assert pl.__version__ == "1.44.1" and sklearn.__version__ == "1.7.1", (
    "install polars==1.44.1 and scikit-learn==1.7.1"
)

In [ ]:
KEYS = [IndCQC.location_id, IndCQC.cqc_location_import_date]
LOCATION = IndCQC.location_id
DATE = IndCQC.cqc_location_import_date
SERVICE = IndCQC.primary_service_type
KNOWN = IndCQC.ascwds_filled_posts_dedup_clean
PRED = IndCQC.prediction
FOLD = Eval.fold
BAND = Eval.size_band

CARE_HOME_MODEL = IndCQC.care_home_model
NON_RES_WITH = IndCQC.non_res_with_dormancy_model
NON_RES_WITHOUT = IndCQC.non_res_without_dormancy_model
NON_RES_COMBINED = IndCQC.non_res_combined_model
SERIES = [CARE_HOME_MODEL, NON_RES_WITH, NON_RES_WITHOUT, NON_RES_COMBINED]

DATASETS = [
    "01_filled_posts_02_cleaned",
    "01_filled_posts_03_imputed",
    *[
        f"01_filled_posts_04_features_{name}"
        for name in (CARE_HOME_MODEL, NON_RES_WITH, NON_RES_WITHOUT)
    ],
]


def dataset_location(dataset: str):
    if LOCAL_ROOT:
        return Path(LOCAL_ROOT) / "datasets" / f"dataset={dataset}"
    return f"s3://{DATA_BUCKET}/domain=03_ind_cqc/dataset={dataset}/"


def scan(dataset: str) -> pl.LazyFrame:
    return polars_utils.scan_parquet(dataset_location(dataset))


def source_files(dataset: str) -> list[dict]:
    # Used to spot a dataset changing between the baseline and a later run.
    if LOCAL_ROOT:
        return [
            {"key": f.name, "size": f.stat().st_size, "last_modified": f.stat().st_mtime}
            for f in sorted(dataset_location(dataset).glob("*.parquet"))
        ]
    pages = (
        boto3.client("s3")
        .get_paginator("list_objects_v2")
        .paginate(Bucket=DATA_BUCKET, Prefix=f"domain=03_ind_cqc/dataset={dataset}/")
    )
    return [
        {"key": o["Key"], "size": o["Size"], "last_modified": o["LastModified"].isoformat()}
        for page in pages
        for o in page.get("Contents", [])
    ]


def write_result(name: str, content: bytes, run_label: str = RUN_LABEL) -> None:
    if LOCAL_ROOT:
        path = Path(LOCAL_ROOT) / "results" / run_label / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
        return
    bucket, key = RESULTS_PREFIX.removeprefix("s3://").split("/", 1)
    boto3.client("s3").put_object(Bucket=bucket, Key=f"{key}{run_label}/{name}", Body=content)


def read_result(run_label: str, name: str) -> bytes:
    if LOCAL_ROOT:
        return (Path(LOCAL_ROOT) / "results" / run_label / name).read_bytes()
    bucket, key = RESULTS_PREFIX.removeprefix("s3://").split("/", 1)
    return boto3.client("s3").get_object(Bucket=bucket, Key=f"{key}{run_label}/{name}")["Body"].read()


def write_parquet_result(name: str, df: pl.DataFrame) -> None:
    buffer = io.BytesIO()
    df.write_parquet(buffer)
    write_result(name, buffer.getvalue())


def read_parquet_result(run_label: str, name: str) -> pl.DataFrame:
    return pl.read_parquet(io.BytesIO(read_result(run_label, name)))


# Self-check: one read and one write, so a permissions problem shows up now.
print("test read, cleaned rows:", scan(DATASETS[0]).select(pl.len()).collect().item())
write_result("_write_test.txt", b"ok")
print("test write ok")


# Pre-flight: fail now, not after an hour of fitting.
assert "model-eval" not in RESULTS_PREFIX and DATA_BUCKET not in RESULTS_PREFIX, "results must go to a persistent prefix"
if BASELINE_RUN_LABEL:
    assert RUN_LABEL != BASELINE_RUN_LABEL, "RUN_LABEL would overwrite the baseline"
    for _name in ("results.parquet", "folds.parquet", "jitter_sample.parquet", "run_info.json"):
        read_result(BASELINE_RUN_LABEL, _name)  # raises if the baseline is missing
print(f"variant {VARIANT!r}, run {RUN_LABEL!r}, baseline {BASELINE_RUN_LABEL!r}, commit {COMMIT}")


## 2. Load and check

- Features: the three production feature datasets (lazy, nothing read until needed).
- Known values from `02_cleaned`; service, CSSR, size bands and the combine step's columns from `03_imputed`.

In [ ]:
SPECS = {name: dict(model_registry[name]) for name in (CARE_HOME_MODEL, NON_RES_WITH, NON_RES_WITHOUT)}
FEATURES = {name: scan(f"01_filled_posts_04_features_{name}") for name in SPECS}

cleaned_lf = scan("01_filled_posts_02_cleaned").select(*KEYS, KNOWN)
imputed_lf = scan("01_filled_posts_03_imputed").select(
    *KEYS,
    IndCQC.care_home,
    SERVICE,
    IndCQC.current_cssr,
    IndCQC.number_of_beds,
    IndCQC.number_of_beds_banded,
    IndCQC.number_of_beds_banded_for_rolling_avg,
    IndCQC.related_location,
    IndCQC.time_registered,
    IndCQC.ascwds_filtering_rule,
    IndCQC.care_home_status_count,
    IndCQC.imputed_filled_post_model,
    IndCQC.imputed_filled_posts_per_bed_ratio_model,
)
CATEGORICAL = [
    IndCQC.care_home,
    SERVICE,
    IndCQC.current_cssr,
    IndCQC.related_location,
    IndCQC.ascwds_filtering_rule,
]
locations_lf = imputed_lf.join(cleaned_lf, on=KEYS, how="left").with_columns(
    pl.col(c).cast(pl.Categorical) for c in CATEGORICAL
)
if CACHE_LOCATIONS:
    # Everything below reads this one table: known values, descriptors and the combine step's columns.
    locations_df = locations_lf.collect()
    print(f"location table cached: {locations_df.height:,} rows, {locations_df.estimated_size() / 2**30:.1f} GiB")
    locations_lf = locations_df.lazy()
imputed_lf = locations_lf.drop(KNOWN)

n_imputed = imputed_lf.select(pl.len()).collect().item()
n_unique_keys = imputed_lf.select(pl.struct(KEYS).n_unique()).collect().item()
n_cleaned = cleaned_lf.select(pl.len()).collect().item()
n_joined = locations_lf.select(pl.len()).collect().item()
print(f"imputed rows {n_imputed:,} ({n_unique_keys:,} unique keys), cleaned rows {n_cleaned:,}, joined {n_joined:,}")
assert n_joined == n_imputed, "joining the cleaned data changed the row count"
dates = locations_lf.select(DATE).unique().sort(DATE).collect()[DATE]
print(f"{dates.len()} import dates, {dates.min()} to {dates.max()}")

In [ ]:
def spec_hash(spec: dict) -> str:
    return hashlib.sha256(",".join(spec[MRKeys.features]).encode()).hexdigest()

In [ ]:
# Experiment definitions. Only the care home and non-res with-dormancy models change; the
# without-dormancy model is left as is so the combined series still runs. Never edit model_registry.py.
ORIGIN = date(2020, 1, 1)  # fixed date for months elapsed
DORMANT_SENTINEL = 999  # time_since_dormant for "never dormant"
TIME_REGISTERED_CAP_MONTHS = 120
INDEXED, CUBED = IndCQC.cqc_location_import_date_indexed, IndCQC.cqc_location_import_date_indexed_cubed
SINCE_DORMANT, TIME_REGISTERED = IndCQC.time_since_dormant, IndCQC.time_registered
MONTHS, MONTHS_CUBED = "months_elapsed", "months_elapsed_cubed"
EVER_DORMANT, MONTHS_DORMANT = "ever_dormant", "months_since_dormant"
TIME_REGISTERED_CAPPED = "time_registered_capped_at_ten_years"


def change(name, drop=(), add=None, params=None):
    """Drop features, add computed ones, and override model params for one model."""
    add = add or {}
    assert all(d in SPECS[name][MRKeys.features] for d in drop), "dropped feature not in the spec"
    FEATURES[name] = FEATURES[name].with_columns(*[expr.alias(column) for column, expr in add.items()])
    features = [f for f in SPECS[name][MRKeys.features] if f not in drop] + list(add)
    SPECS[name] = {
        **SPECS[name],
        MRKeys.features: features,
        MRKeys.model_params: {**SPECS[name][MRKeys.model_params], **(params or {})},
    }


months_elapsed = ((pl.col(DATE).dt.year() - ORIGIN.year) * 12 + pl.col(DATE).dt.month() - ORIGIN.month).cast(pl.Float64)
months_cubed = months_elapsed**3


def dormant(cap):
    never = pl.col(SINCE_DORMANT) == DORMANT_SENTINEL
    return {
        EVER_DORMANT: (~never).cast(pl.Float64),
        MONTHS_DORMANT: pl.when(never).then(0.0).otherwise(pl.col(SINCE_DORMANT).clip(upper_bound=cap)).cast(pl.Float64),
    }


VARIANTS = {
    "none": lambda: None,
    "time_months_ch": lambda: change(CARE_HOME_MODEL, drop=[INDEXED], add={MONTHS: months_elapsed}),
    "time_months_nr": lambda: change(NON_RES_WITH, drop=[INDEXED, CUBED], add={MONTHS: months_elapsed, MONTHS_CUBED: months_cubed}),
    "time_months_nr_nocubic": lambda: change(NON_RES_WITH, drop=[INDEXED, CUBED], add={MONTHS: months_elapsed}),
    "time_index_nr_nocubic": lambda: change(NON_RES_WITH, drop=[CUBED]),
    **{f"dormant_cap{c}_nr": (lambda c=c: change(NON_RES_WITH, drop=[SINCE_DORMANT], add=dormant(c))) for c in (24, 36, 48)},
    "time_registered_cap120_nr": lambda: change(
        NON_RES_WITH,
        drop=[TIME_REGISTERED],
        add={TIME_REGISTERED_CAPPED: pl.col(TIME_REGISTERED).clip(upper_bound=TIME_REGISTERED_CAP_MONTHS)},
    ),
    "maxiter10000_nr": lambda: change(NON_RES_WITH, params={"max_iter": 10_000}),
}
for _a in (0.0001, 0.001, 0.01, 0.1):
    _label = str(_a).replace(".", "p")
    VARIANTS[f"alpha_{_label}_ch"] = lambda a=_a: change(CARE_HOME_MODEL, params={"alpha": a})
    VARIANTS[f"alpha_{_label}_nr"] = lambda a=_a: change(NON_RES_WITH, params={"alpha": a})

VARIANTS[VARIANT]()
for _name in (CARE_HOME_MODEL, NON_RES_WITH, NON_RES_WITHOUT):
    print(_name, SPECS[_name][MRKeys.model_params], len(SPECS[_name][MRKeys.features]), "features")

In [ ]:
from projects._03_independent_cqc._01_filled_posts._04_model.utils import training_utils

# Changed models must still have every feature non-constant in the training rows.
for _name in (CARE_HOME_MODEL, NON_RES_WITH):
    _train = (
        FEATURES[_name]
        .filter(training_utils.is_training_row(SPECS[_name][MRKeys.dependent]))
        .select([pl.col(f).n_unique().alias(f) for f in SPECS[_name][MRKeys.features]])
        .collect()
    )
    print(_name, "constant features:", [c for c in _train.columns if _train[c][0] <= 1] or "none")

### 2a. Cross-check: known values against the filtering rule

In [ ]:
real_rules = [AscwdsFilteringRule.populated, AscwdsFilteringRule.winsorized_beds_ratio_outlier]
rule = pl.col(IndCQC.ascwds_filtering_rule).cast(pl.String).is_in(real_rules)
cross_check_df = (
    locations_lf.group_by(SERVICE)
    .agg(
        pl.col(KNOWN).is_not_null().sum().alias("known_in_cleaned"),
        rule.sum().alias("rule_populated_or_winsorized"),
        (pl.col(KNOWN).is_not_null() & rule).sum().alias("both"),
    )
    .sort(SERVICE)
    .collect()
)
print("known_in_cleaned should equal rule_populated_or_winsorized and both:")
print(cross_check_df)

### 2b. Locations with no target at any date (no ASC-WDS and no PIR)

In [ ]:
TARGET = {True: IndCQC.imputed_filled_posts_per_bed_ratio_model, False: IndCQC.imputed_filled_post_model}


def no_target_locations(care_home: bool) -> pl.DataFrame:
    # Uses the never-submitted flag, keyed on the model target rather than the ASC-WDS value.
    target = TARGET[care_home]
    rows = imputed_lf.filter(is_care_home() if care_home else is_not_care_home()).select(
        LOCATION, IndCQC.care_home_status_count, target
    )
    flagged_lf = evaluation.add_never_submitted_flag(rows, target, LOCATION)
    return (
        flagged_lf.filter(pl.col(Eval.never_submitted) & (pl.col(IndCQC.care_home_status_count) == 1))
        .select(LOCATION)
        .unique()
        .collect()
    )


for care_home, label in ((True, "care home"), (False, "non-res")):
    total = (
        imputed_lf.filter(is_care_home() if care_home else is_not_care_home())
        .select(pl.col(LOCATION).n_unique())
        .collect()
        .item()
    )
    print(f"{label}: {no_target_locations(care_home).height:,} of {total:,} locations have no target at any date")

### 2c. Leakage check: rolling-average group sizes

The rolling-average features average the target over all locations in a group. Care homes: service x bed band. Non-res: service. Leakage only matters where a group has few locations.

In [ ]:
def rolling_average_group_sizes(care_home: bool) -> pl.DataFrame:
    group = [SERVICE, IndCQC.number_of_beds_banded_for_rolling_avg] if care_home else [SERVICE]
    target = TARGET[care_home]
    per_period_lf = (
        imputed_lf.filter((is_care_home() if care_home else is_not_care_home()) & pl.col(target).is_not_null())
        .group_by([*group, DATE])
        .agg(pl.col(LOCATION).n_unique().alias("locations"))
    )
    return (
        per_period_lf.group_by(group)
        .agg(
            pl.col("locations").min().alias("fewest_locations"),
            pl.col("locations").median().alias("median_locations"),
            (pl.col("locations") < SMALL_GROUP_LOCATIONS).sum().alias("periods_below_threshold"),
            pl.len().alias("periods"),
        )
        .sort(group)
        .collect()
    )


print(f"Care homes (small = fewer than {SMALL_GROUP_LOCATIONS} locations in a period):")
print(rolling_average_group_sizes(True))
print("Non-res:")
print(rolling_average_group_sizes(False))

## 3. Folds and out-of-fold predictions

One fold per location across all three datasets. Each model is trained on four folds and predicts the fifth (every row of it).

In [ ]:
all_locations_lf = pl.concat([FEATURES[name].select(LOCATION) for name in SPECS]).unique()
if BASELINE_RUN_LABEL:
    folds_df = read_parquet_result(BASELINE_RUN_LABEL, "folds.parquet")
    baseline_info = json.loads(read_result(BASELINE_RUN_LABEL, "run_info.json"))
    changed = [d for d in DATASETS if source_files(d) != baseline_info["source_files"][d]]
    if changed:
        print("!" * 70)
        print("WARNING: these datasets have changed since the baseline:", changed)
        print("Results are not comparable with the baseline. Re-run the baseline.")
        print("!" * 70)
else:
    folds_df = (
        evaluation.assign_location_folds(all_locations_lf, LOCATION, N_FOLDS, FOLD_SEED)
        .select(LOCATION, FOLD)
        .collect()
    )
folds_lf = folds_df.lazy()
print(folds_df.group_by(FOLD).agg(pl.len().alias("locations")).sort(FOLD))

In [ ]:
if BASELINE_RUN_LABEL:
    print(f"baseline commit {baseline_info['commit']}, this run {COMMIT}.")
    print("The alpha_0p001 runs reproduce the baseline: if they differ, the modelling code or data changed.")
    _base_specs = baseline_info["specs"]
    assert (
        _base_specs[NON_RES_WITHOUT]["features_sha256"] == spec_hash(SPECS[NON_RES_WITHOUT])
        and _base_specs[NON_RES_WITHOUT]["model_params"] == SPECS[NON_RES_WITHOUT][MRKeys.model_params]
    ), "the without-dormancy spec differs from the baseline's"
    for _name in (CARE_HOME_MODEL, NON_RES_WITH):
        _same = (
            _base_specs[_name]["features_sha256"] == spec_hash(SPECS[_name])
            and _base_specs[_name]["model_params"] == SPECS[_name][MRKeys.model_params]
        )
        print(_name, "identical to baseline spec" if _same else "differs from baseline spec")

In [ ]:
from projects._03_independent_cqc._01_filled_posts._04_model.utils import training_utils

FIT_ROWS = []  # one row per fitted model: n_iter_ and exactly-zero coefficients


def record_fit(model_name: str, fit: str, model, features: list[str]) -> None:
    lasso = model[-1]
    zero = [f for f, c in zip(features, lasso.coef_) if c == 0]
    n_iter = int(np.max(lasso.n_iter_))
    FIT_ROWS.append(
        {
            "model": model_name,
            "fit": fit,
            "n_iter": n_iter,
            "max_iter": lasso.max_iter,
            "hit_max_iter": float(n_iter >= lasso.max_iter),
            "zero_coefficients": len(zero),
            "number_of_features": len(features),
            "zero_features": zero,
        }
    )


def predict_out_of_fold_recording_fits(model_name: str, features_lf, spec) -> pl.DataFrame:
    # Same loop as cv.predict_out_of_fold, which does not return its models.
    features_df = cv._add_folds(features_lf, folds_lf, spec).collect()
    predictions = []
    for fold in sorted(features_df[FOLD].unique().to_list()):
        is_held_out = pl.col(FOLD) == fold
        train_df = features_df.filter(~is_held_out & training_utils.is_training_row(spec[MRKeys.dependent]))
        model = cv.fit_model(train_df, spec)
        record_fit(model_name, str(fold), model, spec[MRKeys.features])
        predictions.append(cv._predict_rows(model, features_df.filter(is_held_out), spec))
    return pl.concat(predictions)


oof = {}  # raw model output (a ratio for care homes) for every row, from a model that never saw its location
for name, spec in SPECS.items():
    started = time.time()
    oof[name] = predict_out_of_fold_recording_fits(name, FEATURES[name], spec)
    print(f"{name}: {oof[name].height:,} rows predicted in {time.time() - started:.0f}s")

## 4. Filled posts and the combined non-res model

Care home ratios are multiplied by beds. Predictions go into the combine step **unclipped**, as in production; each final series is then clipped at 1.

In [ ]:
beds_lf = locations_lf.select(*KEYS, IndCQC.number_of_beds)


def combined_non_res(with_dormancy_df: pl.DataFrame, without_dormancy_df: pl.DataFrame) -> pl.LazyFrame:
    # The production combine step, run once over every non-res row.
    non_res_lf = locations_lf.filter(is_not_care_home()).select(
        *KEYS, IndCQC.care_home, IndCQC.related_location, IndCQC.time_registered
    )
    for name, df in ((NON_RES_WITH, with_dormancy_df), (NON_RES_WITHOUT, without_dormancy_df)):
        non_res_lf = model_join.join_model_predictions(non_res_lf, df.lazy(), name, include_run_id=False)
    return (
        combine.combine_non_res_with_and_without_dormancy_models(non_res_lf)
        .select(*KEYS, NON_RES_COMBINED)
        .filter(pl.col(NON_RES_COMBINED).is_not_null())
        .rename({NON_RES_COMBINED: PRED})
    )


posts = {  # filled posts, unclipped
    name: cv.convert_to_filled_posts(oof[name].lazy(), beds_lf, SPECS[name]).collect() for name in SPECS
}
posts[NON_RES_COMBINED] = (
    combined_non_res(posts[NON_RES_WITH], posts[NON_RES_WITHOUT]).join(folds_lf, on=LOCATION, how="left").collect()
)
for name in SERIES:
    print(f"{name}: {posts[name].height:,} rows with a prediction")

## 5. Scoring (known rows only)

Every row scored has both a known value and a prediction, and each series is clipped at 1. Results are per fold and pooled across folds.

In [ ]:
RESULTS = []  # long-format result frames


def to_long(scores_df, model, level, group_columns=(), fold_column=None) -> pl.DataFrame:
    id_columns = [c for c in (*group_columns, fold_column) if c]
    metrics = [c for c in scores_df.columns if c not in id_columns]
    group = (
        pl.concat_str([pl.col(c).cast(pl.String) for c in group_columns], separator=" | ")
        if group_columns
        else pl.lit("all")
    )
    fold = pl.col(fold_column).cast(pl.String) if fold_column else pl.lit("pooled")
    return (
        scores_df.with_columns(group.alias("group"), fold.alias("fold"))
        .select("group", "fold", *[pl.col(m).cast(pl.Float64) for m in metrics])
        .unpivot(index=["group", "fold"], variable_name="metric", value_name="value")
        .with_columns(pl.lit(model).alias("model"), pl.lit(level).alias("level"))
        .select("model", "level", "group", "fold", "metric", "value")
    )


def score_groups(groups_lf, by_columns=None) -> pl.DataFrame:
    scores_lf = evaluation.score_group_totals(groups_lf, PRED, KNOWN, by_columns)
    if by_columns:
        counts_lf = groups_lf.group_by(by_columns).agg(pl.len().alias("number_of_groups"))
        return scores_lf.join(counts_lf, on=by_columns).collect()
    return scores_lf.join(groups_lf.select(pl.len().alias("number_of_groups")), how="cross").collect()


ratio_df = oof[CARE_HOME_MODEL].select(*KEYS, pl.col(PRED).alias("ratio_prediction"))
non_res_bands_lf = (
    cv.add_non_res_size_band(locations_lf.filter(is_not_care_home()), KNOWN, LOCATION)
    .select(LOCATION, BAND)
    .unique()
)


def build_scored(name: str) -> pl.DataFrame:
    lf = posts[name].lazy().join(locations_lf, on=KEYS, how="left")
    if name == CARE_HOME_MODEL:
        lf = lf.join(ratio_df.lazy(), on=KEYS, how="left")
        size_band = pl.col(IndCQC.number_of_beds_banded).cast(pl.Int32).cast(pl.String)
    else:
        lf = lf.join(non_res_bands_lf, on=LOCATION, how="left")
        size_band = pl.col(BAND).cast(pl.String)
    lf = lf.with_columns(
        pl.col(PRED).alias("prediction_unclipped"),
        pl.col(PRED).clip(lower_bound=1.0),
        size_band.alias(BAND),
    )
    lf = evaluation.add_financial_year(lf, DATE)
    return lf.filter(pl.col(KNOWN).is_not_null() & pl.col(PRED).is_not_null()).collect()


scored = {name: build_scored(name) for name in SERIES}

# Coverage: known rows that got a prediction, out of the known rows the series should cover.
known_rows = {
    True: locations_lf.filter(is_care_home() & pl.col(KNOWN).is_not_null()).select(pl.len()).collect().item(),
    False: locations_lf.filter(is_not_care_home() & pl.col(KNOWN).is_not_null()).select(pl.len()).collect().item(),
}
for name in SERIES:
    expected = known_rows[name == CARE_HOME_MODEL]
    coverage_df = pl.DataFrame(
        {
            "known_rows_scored": [scored[name].height],
            "known_rows": [expected],
            "coverage": [scored[name].height / expected],
        }
    )
    RESULTS.append(to_long(coverage_df, name, "coverage"))
    print(f"{name}: scored {scored[name].height:,} of {expected:,} known rows ({scored[name].height / expected:.1%})")

### 5a. Headline: service x CSSR x financial year totals

In [ ]:
GROUPS_5A = [SERVICE, IndCQC.current_cssr, Eval.financial_year]
for name in SERIES:
    lf = scored[name].lazy()
    by_fold_lf = evaluation.aggregate_totals_by_group(lf, PRED, KNOWN, [FOLD, *GROUPS_5A])
    pooled_lf = evaluation.aggregate_totals_by_group(lf, PRED, KNOWN, GROUPS_5A)
    RESULTS.append(to_long(score_groups(by_fold_lf, [FOLD]), name, "headline_group_totals", fold_column=FOLD))
    RESULTS.append(to_long(score_groups(pooled_lf), name, "headline_group_totals"))

### 5b. National totals per service per period: bias and its slope per year

In [ ]:
period_bias = {}  # pooled per-period bias, for the charts
for name in SERIES:
    lf = scored[name].lazy()
    slope_lf = lf.filter(pl.col(Eval.financial_year) >= FIRST_FY) if FIRST_FY else lf
    for by_columns in ([FOLD, SERVICE], [SERVICE]):
        period_totals_lf = evaluation.aggregate_totals_by_group(lf, PRED, KNOWN, [*by_columns, DATE])
        totals_scores_lf = evaluation.score_group_totals(period_totals_lf, PRED, KNOWN, by_columns)
        bias_lf = evaluation.calculate_period_bias(slope_lf, PRED, KNOWN, DATE, by_columns)
        slope_scores_lf = evaluation.fit_bias_slope_per_year(bias_lf, KNOWN, DATE, by_columns)
        scores_df = totals_scores_lf.join(slope_scores_lf, on=by_columns).collect()
        fold_column = FOLD if FOLD in by_columns else None
        RESULTS.append(to_long(scores_df, name, "period_totals", [SERVICE], fold_column))
    period_bias[name] = evaluation.calculate_period_bias(lf, PRED, KNOWN, DATE, [SERVICE]).sort(SERVICE, DATE).collect()

### 5c. Size bands (care homes: beds band. Non-res: band of the location's mean known posts)

In [ ]:
for name in SERIES:
    lf = scored[name].lazy()
    RESULTS.append(to_long(evaluation.score_rows(lf, PRED, KNOWN, [FOLD, BAND]).collect(), name, "size_band", [BAND], FOLD))
    RESULTS.append(to_long(evaluation.score_rows(lf, PRED, KNOWN, [BAND]).collect(), name, "size_band", [BAND]))

### 5d. Row level

- `row_level_posts`: in filled posts, clipped at 1.
- `row_level_metadata_scale`: as `metadata.json` (`calculate_metrics`): care home R2 and RMSE on the per-bed ratio, raw predictions.

In [ ]:
def metadata_scale_metrics(df: pl.DataFrame, name: str) -> dict:
    if name == CARE_HOME_MODEL:
        df = df.filter(pl.col(IndCQC.number_of_beds).is_not_null())
        beds = df[IndCQC.number_of_beds].to_numpy()
        return model_utils.calculate_metrics(
            (df[KNOWN] / df[IndCQC.number_of_beds]).to_numpy(), df["ratio_prediction"].to_numpy(), name, beds
        )
    return model_utils.calculate_metrics(df[KNOWN].to_numpy(), df["prediction_unclipped"].to_numpy(), name)


for name in SERIES:
    lf = scored[name].lazy()
    RESULTS.append(to_long(evaluation.score_rows(lf, PRED, KNOWN, [FOLD]).collect(), name, "row_level_posts", fold_column=FOLD))
    RESULTS.append(to_long(evaluation.score_rows(lf, PRED, KNOWN).collect(), name, "row_level_posts"))
    for fold in [*sorted(folds_df[FOLD].unique().to_list()), None]:
        df = scored[name] if fold is None else scored[name].filter(pl.col(FOLD) == fold)
        metrics_df = pl.DataFrame({k: [v] for k, v in metadata_scale_metrics(df, name).items()})
        RESULTS.append(
            to_long(
                metrics_df.with_columns(pl.lit(fold).alias(FOLD)) if fold is not None else metrics_df,
                name,
                "row_level_metadata_scale",
                fold_column=FOLD if fold is not None else None,
            )
        )

### 5e. Against the imputed target (all rows with a target)

The population the models train on: every month of every location with a target, including PIR-based ones. On known rows the imputed target equals the known value, so this compares predictions with the target on all other rows too. Care homes: ratio x beds.

In [ ]:
TARGET_POSTS = "imputed_target_posts"


def build_target_scored(name: str) -> pl.DataFrame:
    lf = posts[name].lazy().join(locations_lf, on=KEYS, how="left")
    if name == CARE_HOME_MODEL:
        target = pl.col(IndCQC.imputed_filled_posts_per_bed_ratio_model) * pl.col(IndCQC.number_of_beds)
    else:
        target = pl.col(IndCQC.imputed_filled_post_model)
    lf = lf.with_columns(target.cast(pl.Float64).alias(TARGET_POSTS), pl.col(PRED).clip(lower_bound=1.0))
    lf = evaluation.add_financial_year(lf, DATE)
    return (
        lf.filter(pl.col(TARGET_POSTS).is_not_null() & pl.col(PRED).is_not_null())
        .select(*KEYS, FOLD, SERVICE, Eval.financial_year, PRED, TARGET_POSTS)
        .collect()
    )


target_scored = {name: build_target_scored(name) for name in SERIES}
for name in SERIES:
    lf = target_scored[name].lazy()
    print(f"{name}: {target_scored[name].height:,} rows with a target and a prediction")
    RESULTS.append(to_long(evaluation.score_rows(lf, PRED, TARGET_POSTS, [FOLD]).collect(), name, "imputed_target_rows", fold_column=FOLD))
    RESULTS.append(to_long(evaluation.score_rows(lf, PRED, TARGET_POSTS).collect(), name, "imputed_target_rows"))
    slope_lf = lf.filter(pl.col(Eval.financial_year) >= FIRST_FY) if FIRST_FY else lf
    for by_columns in ([FOLD, SERVICE], [SERVICE]):
        bias_lf = evaluation.calculate_period_bias(slope_lf, PRED, TARGET_POSTS, DATE, by_columns)
        slopes_df = evaluation.fit_bias_slope_per_year(bias_lf, TARGET_POSTS, DATE, by_columns).collect()
        RESULTS.append(
            to_long(slopes_df, name, "imputed_target_period_bias", [SERVICE], FOLD if FOLD in by_columns else None)
        )

In [ ]:
def all_results() -> pl.DataFrame:
    # Re-running a section appends its rows again; the latest run of each result wins.
    return pl.concat(RESULTS).unique(
        subset=["model", "level", "group", "fold", "metric"], keep="last", maintain_order=True
    )


def show(level: str, metrics: list[str] | None = None) -> None:
    df = all_results().filter((pl.col("level") == level) & (pl.col("fold") == "pooled"))
    if metrics:
        df = df.filter(pl.col("metric").is_in(metrics))
    print(level)
    print(df.pivot(on="metric", index=["model", "group"], values="value").sort("model", "group"))


show("coverage")
show("headline_group_totals")
show("period_totals")
show("row_level_posts")
show("row_level_metadata_scale")
show("size_band", [Eval.number_of_rows, Eval.bias, Eval.weighted_absolute_percentage_error])
show("imputed_target_rows")
show("imputed_target_period_bias")

## 6. Bridge to production (baseline only)

Held-out predictions scored on the imputed-target rows, as `model_02_train` does. Compare with the latest `metadata.json` from the data branch's resources bucket. Expect close, not identical: the splits differ.

In [ ]:
if RUN_BRIDGE:
    for name, spec in SPECS.items():
        dependent = spec[MRKeys.dependent]
        columns = [*KEYS, IndCQC.care_home_status_count, dependent]
        if name == CARE_HOME_MODEL:
            columns.append(IndCQC.number_of_beds)
        bridge_df = (
            FEATURES[name]
            .select(columns)
            .filter(pl.col(dependent).is_not_null() & (pl.col(IndCQC.care_home_status_count) == 1))
            .join(oof[name].lazy(), on=KEYS, how="inner")
            .collect()
        )
        for fold in [*sorted(folds_df[FOLD].unique().to_list()), None]:
            df = bridge_df if fold is None else bridge_df.filter(pl.col(FOLD) == fold)
            beds = df[IndCQC.number_of_beds].to_numpy() if name == CARE_HOME_MODEL else None
            metrics = model_utils.calculate_metrics(df[dependent].to_numpy(), df[PRED].to_numpy(), name, beds)
            metrics_df = pl.DataFrame({k: [v] for k, v in metrics.items()})
            if fold is not None:
                metrics_df = metrics_df.with_columns(pl.lit(fold).alias(FOLD))
            RESULTS.append(to_long(metrics_df, name, "bridge", fold_column=FOLD if fold is not None else None))
        if LOCAL_ROOT:
            continue
        try:
            root = paths.generate_model_path(DATA_BUCKET, name, spec[MRKeys.version])
            metadata = versioning.load_metadata(root, versioning.get_run_number(root))
            production_df = pl.DataFrame({k: [float(v)] for k, v in metadata["metrics"].items()})
            RESULTS.append(to_long(production_df, name, "bridge_production"))
        except Exception as error:
            print(f"no production metadata for {name}: {error}")
    show("bridge")
    show("bridge_production")

## 7. Jumpiness

Models are trained on all rows (as production). The sample is locations with no target at any date: no ASC-WDS and no PIR, so their published estimate comes from the model. One non-res sample is drawn for both non-res models and the combined series. Jumpiness is the mean absolute change in `log(prediction)` between consecutive periods, so about the average % change.

In [ ]:
def draw_sample(candidates_df: pl.DataFrame, series_label: str) -> pl.DataFrame:
    ids = sorted(candidates_df[LOCATION].to_list())
    chosen = np.random.default_rng(FOLD_SEED).choice(ids, size=min(len(ids), JITTER_SAMPLE_SIZE), replace=False)
    return pl.DataFrame({"series": [series_label] * len(chosen), LOCATION: sorted(chosen.tolist())})


def locations_in(names: list[str]) -> pl.DataFrame:
    return pl.concat([FEATURES[name].select(LOCATION) for name in names]).unique().collect()


if BASELINE_RUN_LABEL:
    sample_df = read_parquet_result(BASELINE_RUN_LABEL, "jitter_sample.parquet")
else:
    sample_df = pl.concat(
        [
            draw_sample(no_target_locations(True).join(locations_in([CARE_HOME_MODEL]), on=LOCATION), "care_home"),
            draw_sample(
                no_target_locations(False).join(locations_in([NON_RES_WITH, NON_RES_WITHOUT]), on=LOCATION),
                "non_res",
            ),
        ]
    )
print(sample_df.group_by("series").agg(pl.len().alias("locations")))

In [ ]:
all_models = {name: cv.fit_on_all_rows(FEATURES[name], spec) for name, spec in SPECS.items()}
all_posts = {  # every row, predicted in fold-sized chunks
    name: cv.convert_to_filled_posts(
        cv.predict_in_chunks(all_models[name], FEATURES[name], folds_lf, SPECS[name]).lazy(), beds_lf, SPECS[name]
    ).collect()
    for name in SPECS
}
all_posts[NON_RES_COMBINED] = combined_non_res(all_posts[NON_RES_WITH], all_posts[NON_RES_WITHOUT]).collect()

LOG = "log_prediction"
for name in SERIES:
    label = "care_home" if name == CARE_HOME_MODEL else "non_res"
    ids_lf = sample_df.filter(pl.col("series") == label).select(LOCATION).lazy()
    sample_lf = (
        all_posts[name]
        .lazy()
        .join(ids_lf, on=LOCATION, how="inner")
        .with_columns(pl.col(PRED).cast(pl.Float64).clip(lower_bound=1.0).log().alias(LOG))
    )
    jumpiness_df = evaluation.mean_period_to_period_change(sample_lf, [LOG], [LOCATION], DATE).collect()
    counts_df = sample_lf.select(pl.col(LOCATION).n_unique().alias("locations"), pl.len().alias("rows")).collect()
    scores_df = jumpiness_df.drop(Eval.column_name).hstack(counts_df)
    RESULTS.append(to_long(scores_df, name, "jumpiness"))
show("jumpiness")

In [ ]:
# Fits on all rows (as production): n_iter_ and exactly-zero coefficients.
for name, model in all_models.items():
    record_fit(name, "all_rows", model, SPECS[name][MRKeys.features])
fit_df = pl.DataFrame(FIT_ROWS)
for name in SPECS:
    RESULTS.append(
        to_long(fit_df.filter(pl.col("model") == name).drop("model", "zero_features"), name, "fit_diagnostics", fold_column="fit")
    )
write_result("fit_diagnostics.json", json.dumps(FIT_ROWS, indent=2).encode("utf-8"))
print(fit_df.drop("zero_features"))
print("features with an exactly-zero coefficient, fitted on all rows:")
for row in FIT_ROWS:
    if row["fit"] == "all_rows":
        print(f"  {row['model']}: {row['zero_features'] or 'none'}")
if fit_df["hit_max_iter"].sum():
    print("WARNING: some fits stopped at max_iter, so are not converged")

## 8. Charts (held-out known rows only)

Monthly mean per location: known against predicted, on the same rows. Care homes also on the per-bed ratio. Then national totals and the per-period bias for each service.

In [ ]:
def label_financial_years(ax, dates) -> None:
    ticks = [date(y, 4, 1) for y in range(min(dates).year, max(dates).year + 1) if min(dates) <= date(y, 4, 1) <= max(dates)]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{t.year}/{str(t.year + 1)[-2:]}" for t in ticks], rotation=45)


def monthly_means(df: pl.DataFrame, known: str, predicted: str) -> pl.DataFrame:
    return (
        df.group_by(DATE)
        .agg(pl.col(known).mean().alias("known"), pl.col(predicted).mean().alias("predicted"))
        .sort(DATE)
    )


def plot_monthly(ax, df, known, predicted, title, known_label="known"):
    means = monthly_means(df, known, predicted)
    ax.plot(means[DATE], means["known"], label=known_label)
    ax.plot(means[DATE], means["predicted"], label="held-out prediction")
    ax.set_title(title)
    ax.grid(True)
    ax.legend()
    label_financial_years(ax, means[DATE].to_list())


care_home_df = scored[CARE_HOME_MODEL].with_columns(
    (pl.col(KNOWN) / pl.col(IndCQC.number_of_beds)).alias("known_ratio")
)
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for i, service in enumerate(["Care home without nursing", "Care home with nursing"]):
    service_df = care_home_df.filter(pl.col(SERVICE) == service)
    plot_monthly(axes[0][i], service_df, KNOWN, PRED, f"{service}: mean filled posts")
    plot_monthly(axes[1][i], service_df, "known_ratio", "ratio_prediction", f"{service}: mean posts per bed")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, name in zip(axes, [NON_RES_WITH, NON_RES_WITHOUT, NON_RES_COMBINED]):
    plot_monthly(ax, scored[name], KNOWN, PRED, f"{name}: mean filled posts")
plt.tight_layout()
plt.show()

Against the imputed target: every row with a target, so each location counts for all its months.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, service in zip(axes, ["Care home without nursing", "Care home with nursing"]):
    df = target_scored[CARE_HOME_MODEL].filter(pl.col(SERVICE) == service)
    plot_monthly(ax, df, TARGET_POSTS, PRED, f"{service}: mean filled posts", "imputed target")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, name in zip(axes, [NON_RES_WITH, NON_RES_WITHOUT, NON_RES_COMBINED]):
    plot_monthly(ax, target_scored[name], TARGET_POSTS, PRED, f"{name}: mean filled posts", "imputed target")
plt.tight_layout()
plt.show()

In [ ]:
for name in SERIES:
    services = period_bias[name][SERVICE].unique().sort().to_list()
    fig, axes = plt.subplots(2, len(services), figsize=(6 * len(services), 8), squeeze=False)
    for i, service in enumerate(services):
        df = period_bias[name].filter(pl.col(SERVICE) == service)
        axes[0][i].plot(df[DATE], df[KNOWN], label="known total")
        axes[0][i].plot(df[DATE], df[PRED], label="held-out predicted total")
        axes[0][i].set_title(f"{name}: {service}")
        axes[0][i].legend()
        axes[1][i].plot(df[DATE], df[Eval.bias])
        axes[1][i].axhline(0, color="black", linewidth=0.8)
        axes[1][i].set_title("per-period bias: (predicted - known) / known")
        for ax in (axes[0][i], axes[1][i]):
            ax.grid(True)
            label_financial_years(ax, df[DATE].to_list())
    plt.tight_layout()
    plt.show()

## 9. Save

In [ ]:
def spec_info(spec: dict) -> dict:
    features = spec[MRKeys.features]
    return {
        "model_type": spec[MRKeys.model_type],
        "model_params": spec[MRKeys.model_params],
        "dependent": spec[MRKeys.dependent],
        "features": features,
        "features_sha256": hashlib.sha256(",".join(features).encode()).hexdigest(),
    }


results_df = all_results().with_columns(pl.lit(RUN_LABEL).alias("run_label"))
run_info = {
    "run_label": RUN_LABEL,
    "baseline_run_label": BASELINE_RUN_LABEL,
    "created": datetime.now(timezone.utc).isoformat(),
    "commit": COMMIT,
    "variant": VARIANT,
    "python": platform.python_version(),
    "polars": pl.__version__,
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "settings": {
        "fold_seed": FOLD_SEED,
        "n_folds": N_FOLDS,
        "jitter_sample_size": JITTER_SAMPLE_SIZE,
        "first_fy": FIRST_FY,
        "data_bucket": DATA_BUCKET,
    },
    "specs": {name: spec_info(spec) for name, spec in SPECS.items()},
    "source_files": {d: source_files(d) for d in DATASETS},
}

write_parquet_result("results.parquet", results_df)
write_result("run_info.json", json.dumps(run_info, indent=2, default=str).encode("utf-8"))
write_parquet_result("folds.parquet", folds_df)
write_parquet_result("jitter_sample.parquet", sample_df)
write_parquet_result(
    "period_bias.parquet",
    pl.concat([df.with_columns(pl.lit(name).alias("model")) for name, df in period_bias.items()]),
)
print(f"saved {results_df.height:,} result rows for run '{RUN_LABEL}'")

## Running an experiment

1. Set `VARIANT`, `RUN_LABEL` and `BASELINE_RUN_LABEL` in the settings cell (three modes are described there).
2. Run top to bottom. Folds and the jitter sample come from the baseline; a warning shows if the source datasets changed.
3. Compare with `compare.ipynb`.

Results land in `RESULTS_PREFIX/<RUN_LABEL>/`. `fit_diagnostics` has `n_iter_` per fit; `fit_diagnostics.json` lists exactly-zero coefficients.